# Middleware for LangChain Agents

This notebook explains what middleware is, why it helps control agents more tightly, and provides code examples for common middleware patterns.

## What is middleware?

Middleware is a layer of reusable code that sits between the user request and the agent or model execution. It can inspect, modify, or block requests and responses, giving you finer control over how the agent behaves.

In AI engineering, middleware is useful for logging, validation, prompt transformation, fallback behavior, and safety checks without changing the core agent logic.

## Why use middleware with agents?

Tight control of an agent means you can add cross-cutting behavior before and after each agent call. Instead of modifying the agent internals, middleware wraps agent execution and adds behavior in a composable way.

Middleware is ideal for these agent control concerns:
1. Tracking agent behavior (logging, analytics, debugging)
2. Transforming prompts, tool selection, output formatting
3. Adding retries, fallback, early termination logic
4. Applying rate limits, guardrails, and PII detection

## Middleware pattern

A simple middleware pattern wraps an agent call with pre- and post-processing logic. The agent executor is treated like a function, and middleware can be chained.

In [ ]:
from typing import Any, Callable, Dict, List

AgentFunction = Callable[[Dict[str, Any]], Dict[str, Any]]

class AgentMiddleware:
    def __init__(self, handler: AgentFunction):
        self.handler = handler

    def __call__(self, inputs: Dict[str, Any]) -> Dict[str, Any]:
        return self.handler(inputs)

def compose_middlewares(middlewares: List[AgentMiddleware], base_handler: AgentFunction) -> AgentFunction:
    handler = base_handler
    for middleware in reversed(middlewares):
        previous = handler
        handler = lambda inputs, mw=middleware, prev=previous: mw(prev)(inputs)
    return handler

## 1. Tracking agent behavior

Use middleware to log requests and responses, capture analytics, or trace debugging details. This is the first line of visibility into agent decisions.

In [ ]:
class LoggingMiddleware(AgentMiddleware):
    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            print("[LOG] Agent input:", inputs)
            result = handler(inputs)
            print("[LOG] Agent output:", result)
            return result
        return wrapped

def dummy_agent(inputs: Dict[str, Any]) -> Dict[str, Any]:
    return {"output": f"Echo: {inputs.get('input')}"}

logging_mw = LoggingMiddleware(lambda handler: handler)
wrapped_agent = logging_mw(dummy_agent)
print(wrapped_agent({"input": "Hello AI"}))

## 2. Transforming prompts, tool selection, output formatting

Middleware can rewrite prompts, choose which tools are available, and normalize the final output. These transformations allow the agent to receive safer inputs and return cleaner results.

In [ ]:
class PromptTransformMiddleware(AgentMiddleware):
    def __init__(self, transform: Callable[[str], str]):
        self.transform = transform

    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            prompt = inputs.get("prompt", "")
            inputs["prompt"] = self.transform(prompt)
            return handler(inputs)
        return wrapped

class OutputFormatMiddleware(AgentMiddleware):
    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            result = handler(inputs)
            if "output" in result:
                result["formatted_output"] = result["output"].strip()
            return result
        return wrapped

def basic_agent(inputs: Dict[str, Any]) -> Dict[str, Any]:
    return {"output": inputs.get("prompt", "") + " [processed]"}

prompt_middleware = PromptTransformMiddleware(lambda prompt: prompt + ' Please answer clearly.')
format_middleware = OutputFormatMiddleware(lambda handler: handler)
wrapped = prompt_middleware(format_middleware(basic_agent))
print(wrapped({"prompt": "What is middleware?"}))

## 3. Retries, fallback, early termination

Retries and fallback logic help make agents robust when external services fail or when the response is not satisfactory. Early termination can stop the agent if a guardrail is triggered.

In [ ]:
class RetryMiddleware(AgentMiddleware):
    def __init__(self, retries: int = 2):
        self.retries = retries

    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            last_exception = None
            for attempt in range(1, self.retries + 1):
                {
                    "cell_type": "code",
                    "metadata": {
                        "language": "python"
                    },
                    "source": [
                        "# Version-checked example: prefer built-in LangChain SummarizationMiddleware, fallback otherwise\n",
                        "import importlib\n",
                        "\n",
                        "def has_module(name):\n",
                        "    try:\n",
                        "        importlib.import_module(name)\n",
                        "        return True\n",
                        "    except Exception:\n",
                        "        return False\n",
                        "\n",
                        "print('langchain installed:', has_module('langchain'))\n",
                        "\n",
                        "# Attempt to use built-in SummarizationMiddleware + create_agent if available\n",
                        "try:\n",
                        "    from langchain.agents import create_agent\n",
                        "    from langchain.agents.middleware import SummarizationMiddleware as LC_SummarizationMiddleware\n",
                        "    try:\n",
                        "        from langgraph.checkpoint.memory import InMemorySaver\n",
                        "        checkpointer = InMemorySaver()\n",
                        "    except Exception:\n",
                        "        checkpointer = None\n",
                        "    print('Using built-in SummarizationMiddleware from langchain.agents.middleware')\n",
                        "    # Build agent with middleware (note: exact args may vary by LangChain version)\n",
                        "    try:\n",
                        "        mw = LC_SummarizationMiddleware(model='gpt-4o-mini', trigger=('messages', 10), keep=('messages', 5))\n",
                        "        agent = create_agent(model='gpt-4o-mini', checkpointer=checkpointer, middleware=[mw])\n",
                        "        print('Created agent using built-in middleware (not executing a run).')\n",
                        "    except TypeError as te:\n",
                        "        print('create_agent or middleware signature mismatch:', te)\n",
                        "except Exception as e:\n",
                        "    print('Built-in middleware not available or failed to construct:', e)\n",
                        "    print('Falling back to the notebook SummarizationMiddleware implementation (uses LangChain summarizer if installed).')\n",
                        "    try:\n",
                        "        # `mw`, `agent_with_history`, and `long_messages` are defined in the previous example cell; reuse them here.\n",
                        "        wrapped_agent = mw(agent_with_history)\n",
                        "        out = wrapped_agent({'messages': long_messages})\n",
                        "        print('Fallback wrapped agent output:', out)\n",
                        "    except Exception as ex:\n",
                        "        print('Fallback example failed to run:', ex)\n"
                    ]
                },
                {
                    "cell_type": "markdown",
                    "id": "#VSC-91a1b175",
                    "metadata": {
                        "language": "markdown"
                    },
                    "source": [
                        "## Summary",
                        "",
                        "Middleware gives you a clean way to add cross-cutting controls to LangChain agents. It helps you observe behavior, transform inputs and outputs, handle failures, and enforce safety without changing the core agent implementation."
                    ]
                }
    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            try:
                return handler(inputs)
            except Exception:
                return {"output": self.fallback_response}
        return wrapped

def flaky_agent(inputs: Dict[str, Any]) -> Dict[str, Any]:
    raise RuntimeError("Model call failed")

handler = RetryMiddleware(retries=3)(FallbackMiddleware("Sorry, I cannot answer right now.")(flaky_agent))
print(handler({"input": "Test"}))

## 4. Rate limits, guardrails, and PII detection

Middleware can enforce request quotas, prevent disallowed content, and detect sensitive data before the agent runs. This is critical for safety and compliance.

In [ ]:
import time
from collections import deque

class RateLimitMiddleware(AgentMiddleware):
    def __init__(self, max_calls: int, period_seconds: int):
        self.max_calls = max_calls
        self.period_seconds = period_seconds
        self.calls = deque()

    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            now = time.time()
            while self.calls and now - self.calls[0] > self.period_seconds:
                self.calls.popleft()
            if len(self.calls) >= self.max_calls:
                return {"output": "Rate limit exceeded. Try again later."}
            self.calls.append(now)
            return handler(inputs)
        return wrapped

class PiiMiddleware(AgentMiddleware):
    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            prompt = inputs.get("prompt", "
,
SSN" in prompt or "credit card" in prompt:
                return {"output": "Potential sensitive data detected. Request blocked."}
            return handler(inputs)
        return wrapped

def simple_agent(inputs: Dict[str, Any]) -> Dict[str, Any]:
    return {"output": "OK"}

guarded_agent = RateLimitMiddleware(2, 10)(PiiMiddleware(lambda handler: handler)(simple_agent))
print(guarded_agent({"prompt": "Hello"}))
print(guarded_agent({"prompt": "Hello again"}))
print(guarded_agent({"prompt": "Send SSN details"}))

### Built-in middleware
After some chunks of messages, the agent is going to summarize the messages, when approacing token limits. (Summarization Middleware).


In [ ]:
from typing import Callable
import textwrap


class SummarizationMiddleware(AgentMiddleware):
    """Summarization middleware compresses conversation history when it grows
    beyond a configured threshold. This implementation attempts to use
    LangChain's summarization tools (LLM-based) and falls back to a
    lightweight summarizer if LangChain isn't available.
    """

    def __init__(self, summarizer: Callable[[str], str], max_messages: int = 20, max_chars: int = 400):
        self.summarizer = summarizer
        self.max_messages = max_messages
        self.max_chars = max_chars

    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            messages = inputs.get("messages")
            if isinstance(messages, list):
                total_chars = sum(len(m.get("content", "")) for m in messages)
                if len(messages) > self.max_messages or total_chars > self.max_chars:
                    joined = "\n".join(f"{m.get('role','')}: {m.get('content','')}" for m in messages)
                    summary = self.summarizer(joined)
                    inputs["messages"] = [{"role": "system", "content": f"Conversation summary:\n{summary}"}]
            else:
                prompt = inputs.get("prompt", "")
                if len(prompt) > self.max_chars:
                    summary = self.summarizer(prompt)
                    inputs["prompt"] = f"SUMMARY: {summary}\n\nOriginal question: {prompt[:200]}"

            return handler(inputs)

        return wrapped


def simple_summarizer(text: str, max_len: int = 200) -> str:
    """A lightweight summarizer for examples. Use an LLM call in production."""
    return textwrap.shorten(text.replace("\n", " "), width=max_len, placeholder="...")


# Try to use LangChain's summarization chain if available, otherwise fallback
try:
    from langchain.llms import OpenAI
    from langchain.chains.summarize import load_summarize_chain
    from langchain.docstore.document import Document

    def langchain_summarizer(text: str, max_len: int = 200) -> str:
        llm = OpenAI(temperature=0)
        docs = [Document(page_content=text)]
        chain = load_summarize_chain(llm, chain_type="map_reduce")
        return chain.run(docs)

    active_summarizer = langchain_summarizer
except Exception:
    active_summarizer = simple_summarizer


# Example usage
def agent_with_history(inputs: Dict[str, Any]) -> Dict[str, Any]:
    messages = inputs.get("messages")
    if isinstance(messages, list) and messages:
        return {"output": f"Processed {len(messages)} messages. Last: {messages[-1].get('content','')[:60]}"}
    return {"output": inputs.get("prompt", "No input")}


long_messages = [{"role": "user", "content": f"Turn {i}: This is a long message about a topic. " * 5} for i in range(30)]
mw = SummarizationMiddleware(lambda t: active_summarizer(t, max_len=250), max_messages=10, max_chars=1000)
wrapped_agent = mw(agent_with_history)
result = wrapped_agent({"messages": long_messages})
print(result)

## 5. Message Types: HumanMessage and SystemMessage

Messages are the fundamental units of communication in LangChain. When middleware intercepts agent calls, it often works with message lists. Understanding message types helps you build and manipulate conversations in middleware.

### What is HumanMessage?
A `HumanMessage` represents input from the user or external source. It is the main way users communicate with the agent.

### What is SystemMessage?
A `SystemMessage` contains instructions or context for the agent (e.g., role, behavior guidelines, conversation history summaries).

### Why it matters for middleware
- Middleware can inspect and modify messages before they reach the agent.
- You can inject `SystemMessage` instances (e.g., conversation summaries) to guide agent behavior.
- Logging middleware can track `HumanMessage` and `AIMessage` sequences.
- Transformation middleware can rewrite user input or inject system instructions.

In [ ]:
# HumanMessage and SystemMessage in middleware
try:
    from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
except ImportError:
    # Fallback for older langchain versions
    try:
        from langchain.schema import HumanMessage, SystemMessage, AIMessage
    except ImportError:
        print('LangChain core messages not installed. Using mock definitions for demo.')
        class HumanMessage(dict):
            def __init__(self, content):
                super().__init__(role='user', content=content)
        class SystemMessage(dict):
            def __init__(self, content):
                super().__init__(role='system', content=content)
        class AIMessage(dict):
            def __init__(self, content):
                super().__init__(role='assistant', content=content)

# Example: MessageTransformMiddleware that injects system context
class MessageTransformMiddleware(AgentMiddleware):
    """Middleware that inspects HumanMessage and injects SystemMessage context."""
    def __init__(self, system_instruction: str):
        self.system_instruction = system_instruction

    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            messages = inputs.get('messages', [])
            # Inspect: find HumanMessage instances
            human_count = sum(1 for m in messages if isinstance(m, dict) and m.get('role') == 'user')
            print(f'[MessageTransform] Found {human_count} HumanMessage(s)')
            # Inject: prepend a SystemMessage with instructions
            if isinstance(messages, list):
                system_msg = {'role': 'system', 'content': self.system_instruction}
                if system_msg not in messages:
                    inputs['messages'] = [system_msg] + messages
            return handler(inputs)
        return wrapped

# Example usage
messages = [
    {'role': 'user', 'content': 'What is the weather?'},
    {'role': 'assistant', 'content': 'I don\'t have real-time weather data.'}
]

def echo_agent(inputs: Dict[str, Any]) -> Dict[str, Any]:
    msgs = inputs.get('messages', [])
    return {'output': f'Agent saw {len(msgs)} messages'}

mw = MessageTransformMiddleware('You are a helpful AI assistant. Be concise.')
wrapped = mw(echo_agent)
result = wrapped({'messages': messages})
print('Result:', result)
print('Updated messages include system instruction at the top.')

In [ ]:
# Advanced example: LoggingMiddleware that tracks message types
class MessageLoggingMiddleware(AgentMiddleware):
    """Logs message types and counts in the conversation."""
    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            messages = inputs.get('messages', [])
            # Analyze message composition
            stats = {
                'human': sum(1 for m in messages if isinstance(m, dict) and m.get('role') == 'user'),
                'system': sum(1 for m in messages if isinstance(m, dict) and m.get('role') == 'system'),
                'assistant': sum(1 for m in messages if isinstance(m, dict) and m.get('role') == 'assistant')
            }
            print(f'[MessageLogger] Before: {stats}')
            result = handler(inputs)
            print(f'[MessageLogger] After: Agent returned {result}')
            return result
        return wrapped

# Test MessageLoggingMiddleware
conv = [
    {'role': 'system', 'content': 'You are helpful.'},
    {'role': 'user', 'content': 'Hello!'},
    {'role': 'assistant', 'content': 'Hi there!'},
    {'role': 'user', 'content': 'How are you?'}
]

logging_mw = MessageLoggingMiddleware(lambda h: h)
logged = logging_mw(echo_agent)
output = logged({'messages': conv})
print('Final output:', output)

### Key takeaways

- **HumanMessage**: represents user input; middleware can inspect, validate, or transform it.
- **SystemMessage**: carries instructions; middleware can inject or modify system context.
- **AIMessage**: the agent's response; can be logged or analyzed by middleware.
- Middleware sits at the message level, allowing you to control how conversations flow without modifying the core agent.
- Use `role='user'` (HumanMessage), `role='system'` (SystemMessage), and `role='assistant'` (AIMessage) to structure conversations.